# BESSTIE Sentiment and Sarcasm Dataset Sample

Run the cells in order to download the public training and validation splits of [BESSTIE](https://huggingface.co/datasets/unswnlporg/BESSTIE) for Australian, Indian, and British English. The first run requires internet access; Hugging Face caches the data locally for subsequent runs.

Select the repository's `.venv` Python kernel with the Hugging Face `datasets` package installed before running. This notebook uses the existing environment and does not install packages or train a model.

The dataset includes Google Places reviews and Reddit comments. Each example has independent binary sentiment and sarcasm labels. The official test set is withheld; only training and validation splits are downloaded here.



## Download all three English varieties

The revision is pinned so the schema, split counts, and published license match the release described in [the dataset notes](../datasets.md). The configurations remain separate, preserving their original training and validation boundaries.

In [1]:
from datasets import concatenate_datasets, load_dataset

DATASET_ID = "unswnlporg/BESSTIE"
REVISION = "4f1c0b2a6e2a3bd02631e6ea0b6e0c9eaa09a0e7"
CONFIGS = ("en_AU", "en_IN", "en_UK")

datasets_by_variety = {
    config: load_dataset(DATASET_ID, config, revision=REVISION)
    for config in CONFIGS
}

for config, dataset in datasets_by_variety.items():
    print(
        f"{config}: train={len(dataset['train']):,}, "
        f"validation={len(dataset['validation']):,}"
    )

c:\Users\jlytt\source\repos\GitHub\distilbert-sentiment-analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


en_AU: train=2,709, validation=371
en_IN: train=3,332, validation=454
en_UK: train=2,822, validation=386


## Select a balanced training preview

Combine only the training splits for this preview. Select five non-sarcastic and five sarcastic examples using seed `42`; this deliberately balanced sample does not represent the dataset's natural class proportions.

The integer label meanings come from the author dataset card:

* Sentiment: `0` = negative, `1` = positive. There is no neutral class.
* Sarcasm: `0` = non-sarcastic, `1` = sarcastic. Sarcasm does not automatically imply negative sentiment.

In [2]:
train_data = concatenate_datasets(
    [datasets_by_variety[config]["train"] for config in CONFIGS]
)
sentiment_names = {0: "negative", 1: "positive"}
sarcasm_names = {0: "non-sarcastic", 1: "sarcastic"}

non_sarcastic_examples = (
    train_data.filter(lambda row: row["sarcasm"] == 0)
    .shuffle(seed=42)
    .select(range(5))
)
sarcastic_examples = (
    train_data.filter(lambda row: row["sarcasm"] == 1)
    .shuffle(seed=42)
    .select(range(5))
)
sample_data = concatenate_datasets([non_sarcastic_examples, sarcastic_examples])

In [3]:
for index, row in enumerate(sample_data, start=1):
    print(f"Example {index} | Source: {row['source']} | Variety: {row['variety']}")
    print(f"Text: {row['text']}")
    print(f"Sentiment: {row['sentiment']} ({sentiment_names[row['sentiment']]})")
    print(f"Sarcasm: {row['sarcasm']} ({sarcasm_names[row['sarcasm']]})")
    print("-" * 70)

Example 1 | Source: google | Variety: en-UK
Text: Fantastic service, great atmosphere, good selection of beers. Food was very tasty and portions VERY generous. Chicken was not as crispy as I though( but delicious) and I found the strips and onion rings to be a bit too oily by my partner thought the chicken burger was n't at all. Still-will definitely come again and recommend to others.
Sentiment: 1 (positive)
Sarcasm: 0 (non-sarcastic)
----------------------------------------------------------------------
Example 2 | Source: google | Variety: en-IN
Text: Went for a take away order for 2 plates of dahi vada. After reaching home, realised they had packed only dahi without any vada in it. Upon informing them the same over phone, they arranged for delivery of the dahi vada at home. Delivery was little slow as the owner had to arrange for it ad hoc but highly appreciate her call to deliver it to our home address. Dahi vada tasted good.. Highly impressed with the integrity, handling of custo

## Interpretation limits

Sarcasm prevalence varies by source and English variety, and sarcasm annotation agreement is imperfect. Some Indian English comments contain code-mixing. Explicit markers such as `/s` may provide shortcuts, so this preview is not evidence that a model understands sarcasm.

The sentiment labels are reader annotations, not separate literal and intended sentiment targets. Keep the published validation split separate from training, and do not call it an untouched test after tuning on it.